# Student Performance Prediction System
Google Colab version: download data → train → evaluate → test.

**Target:** G3 < 10 = AT RISK; G3 ≥ 10 = NOT AT RISK. G3 is never used as an input feature.

*Responsible use: predictions are not diagnoses and must not be the sole basis for real student decisions.*

## 1. Install and import

In [ ]:
!pip install -q ucimlrepo

import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

## 2. Retrieve and inspect the dataset

In [ ]:
student = fetch_ucirepo(id=320)
df = pd.concat([student.data.features, student.data.targets], axis=1)
df.to_csv("student_data.csv", index=False)

print("Rows and columns:", df.shape)
display(df.head())
print(df.dtypes)
print("Missing values per column:")
print(df.isnull().sum())
assert {"G1", "G2", "G3"}.issubset(df.columns), "G1, G2, G3 must be present"

## 3. Preprocess and train

In [ ]:
# Target: 1 = at risk (G3 < 10); 0 = not at risk
df = df.drop_duplicates()
y = (df["G3"] < 10).astype(int)

# Never use G3 as a predictor
X = df.drop(columns=["G3"])

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_cols),
    ("categorical", categorical_pipe, categorical_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight="balanced"
    )),
])

model.fit(X_train, y_train)
print("Model trained.")

## 4. Evaluate
Record accuracy, precision, recall and F1. Do not interpret accuracy alone.

In [ ]:
predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print(classification_report(
    y_test, predictions,
    target_names=["Not at risk", "At risk"],
    zero_division=0,
))
print("Confusion matrix (rows = actual, cols = predicted):")
print(confusion_matrix(y_test, predictions))

## 5. Save the model

In [ ]:
joblib.dump(model, "student_model.pkl")
print("Saved student_model.pkl")

## 6. Test predictions on sample records

In [ ]:
sample = X_test.head(5).copy()
sample["Prediction"] = [
    "AT RISK" if p == 1 else "NOT AT RISK"
    for p in model.predict(X_test.head(5))
]
sample[["G1", "G2", "absences", "failures", "Prediction"]]

## 7. Download the generated files
Upload these to your GitHub repository (if your instructor allows), then run the Streamlit app locally with `streamlit run app.py`.

In [ ]:
from google.colab import files
files.download("student_data.csv")
files.download("student_model.pkl")